# LLM Roofline 결과 보기 (벤치마크 실행 없음)

[LLM roofline 노트북](llm_roofline.ipynb)으로 잰 결과(`llm_roofline/results/*.json`)를 불러와 그래프와 표로 본다. **벤치마크를 다시 돌리지 않으므로** GPU·TPU 없이 로컬에서 바로 실행된다.

```bash
uv run --with jupyterlab --with matplotlib --with pandas jupyter lab notebooks/llm_roofline_results.ipynb
```

지금 들어 있는 결과는 같은 모델·같은 크기(`colab` preset, bf16)를 두 하드웨어에서 잰 것이다.

| 파일 | 프레임워크 | 하드웨어 |
|---|---|---|
| `colab_a100.json` | PyTorch | NVIDIA A100-SXM4-40GB |
| `colab_tpu_v5e.json` | JAX | TPU v5e (v5 lite) |

새로 잰 JSON을 `llm_roofline/results/`에 넣으면 아래 셀들이 자동으로 함께 그린다.

## 0. 준비

이 노트북이 있는 저장소를 찾아 작업 디렉터리로 삼는다. 필요한 패키지는 `numpy`, `matplotlib`, `pandas`뿐이다.

In [ ]:
import os, sys, glob, json

def find_repo():
    d = os.getcwd()
    while not os.path.isdir(os.path.join(d, "llm_roofline", "results")):
        if os.path.dirname(d) == d:
            raise FileNotFoundError("저장소(tensor2silicon) 안에서 실행하세요")
        d = os.path.dirname(d)
    return d

REPO = find_repo()
os.chdir(REPO)
sys.path.insert(0, REPO)

import matplotlib.pyplot as plt
import pandas as pd
from llm_roofline.plot import plot_roofline, plot_time_breakdown, summary_table

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
RESULTS = sorted(glob.glob("llm_roofline/results/*.json"))
DATA = {os.path.basename(p).removesuffix(".json"): json.load(open(p)) for p in RESULTS}
print("repo:", REPO)
print("results:", list(DATA))

## 1. 실행 환경과 roofline 천장

roofline 천장은 spec sheet 값이 아니라 **직접 잰 값**이다. 큰 matmul의 TFLOP/s와 1 GiB copy·read 중 빠른 쪽의 GB/s를 쓴다. `ridge`(= peak FLOP/s ÷ peak bytes/s)보다 AI가 작으면 memory-bound, 크면 compute-bound다.

In [ ]:
rows = []
for name, d in DATA.items():
    m = d["meta"]
    (mf, mb), spec = m["peaks_used"], m.get("spec_peaks")
    rows.append({"result": name, "framework": f"{m['framework']} {m['version']}", "device": m["device"],
                 "dtype": m["dtype"], "preset": m["preset"],
                 "measured TFLOP/s": round(mf / 1e12), "measured GB/s": round(mb / 1e9), "ridge (FLOP/B)": round(mf / mb),
                 "spec TFLOP/s": round(spec[0] / 1e12) if spec else None, "spec GB/s": round(spec[1] / 1e9) if spec else None})
pd.DataFrame(rows).set_index("result")

## 2. Roofline 그림

x축은 arithmetic intensity(FLOP/byte), y축은 달성한 TFLOP/s다. 굵은 선이 roofline `min(peak FLOP/s, AI × BW)`이고, 세로선이 ridge point다.

- **prefill**(토큰 2048개): 가중치 matmul(`q,k,v,o_proj`, `gate,up_proj`, `down_proj`, `lm_head`)이 ridge 오른쪽 수평선 근처에 있다. **compute-bound**다.
- **decode**(토큰 1개 + KV cache 2048개): **같은 matmul**이 AI ≈ 8로 떨어져 대각선에 붙는다. **memory-bound**다.
- `LayerNorm`·`softmax`·elementwise 연산은 두 단계 모두 AI ≈ 1 이하라 항상 memory-bound다.
- 같은 모양의 matmul은 점 하나로 합쳤다. FLOPs가 0인 데이터 이동 연산(`embed`, `merge_heads`, KV cache 쓰기)은 log 축에 그릴 수 없어서 표에만 나온다.

In [ ]:
for name, d in DATA.items():
    plot_roofline(d)
    plt.show()

## 3. 시간은 어디에 쓰이나

연산별 시간 × 호출 횟수(층 수 L=8)를 더해 forward 한 번에 각 연산이 차지하는 시간을 본다. 막대 끝의 %는 전체에서 차지하는 비율이다(4% 이상만 표시).

In [ ]:
for name, d in DATA.items():
    plot_time_breakdown(d)
    plt.show()

## 4. 연산별 비교표

두 하드웨어에서 같은 연산을 나란히 놓는다.

- `AI`: FLOP/byte (하드웨어와 무관)
- `%roofline`: roofline 시간 ÷ 측정 시간. 100%에 가까울수록 하드웨어 한계까지 쓴 것이다.
- `measured`: 실제로 더 많이 쓴 자원. roofline 시간이 5 µs 미만이고 %roofline이 25% 미만이면 `latency`(launch·dispatch 지연이 지배)다.

100%를 조금 넘는 값은 측정 천장의 오차이거나, 데이터가 캐시에 남아 있어서 생긴다(예: A100 `embed`의 118%는 gather한 행 32 MB가 L2 40 MB에 남은 경우).

In [ ]:
tables = []
for name, d in DATA.items():
    t = pd.DataFrame(summary_table(d))
    t["result"] = name
    tables.append(t)
ops = pd.concat(tables)

order = {op: i for i, op in enumerate(dict.fromkeys(ops["op"]))}   # 실행 순서 유지
compare = (ops.pivot_table(index=["phase", "op"], columns="result", values=["us", "%roofline", "measured"], aggfunc="first")
              .swaplevel(0, 1, axis=1).sort_index(axis=1))
compare = compare.reindex(sorted(compare.index, key=lambda k: (k[0] != "prefill", order[k[1]])))
compare.insert(0, "AI", ops.drop_duplicates(["phase", "op"]).set_index(["phase", "op"])["AI"])
compare

## 5. 9개 matmul만 보기: prefill → decode에서 병목이 바뀐다

In [ ]:
mm = ops[ops["#"] != ""].copy()
mm["label"] = mm["#"].astype(str) + " " + mm["op"]
view = mm.pivot_table(index="label", columns=["result", "phase"], values="measured", aggfunc="first")
view = view.reindex(index=sorted(view.index, key=lambda s: int(s.split()[0])),
                    columns=[(r, ph) for r in DATA for ph in ("prefill", "decode")])
view

## 6. 모델 전체와 attention

- **sum of ops**: 따로 잰 연산 시간 × 호출 횟수의 합. 연산 사이에 fusion이 없을 때의 시간이다.
- **eager / eager_graph / compiled** (PyTorch): 연산 하나씩 launch / 같은 kernel을 CUDA graph로 launch / `torch.compile`
- **jit** (JAX): `jax.jit`으로 모델 전체를 XLA 프로그램 하나로 컴파일
- **attention**: 모델 안의 unfused(`qk → mask+softmax → pv`)와 fused kernel 비교

In [ ]:
rows = []
for name, d in DATA.items():
    for ph, v in d["phases"].items():
        row = {"result": name, "phase": ph, "sum of ops (ms)": round(v["summary"]["sum_of_ops_s"] * 1e3, 2)}
        for k, x in v["model"].items():
            if k.endswith("_s") and k != "compile_time_s":
                row[f"{k.removesuffix('_s')} (ms)"] = round(x * 1e3, 2)
        a = v["attention"]
        row.update({"attn unfused (us)": round(a["unfused_s"] * 1e6, 1), "attn fused (us)": round(a["fused_s"] * 1e6, 1),
                    "fused speedup": round(a["unfused_s"] / a["fused_s"], 2), "fused impl": a["fused_impl"]})
        rows.append(row)
pd.DataFrame(rows).set_index(["result", "phase"])

## 7. 시간 비중: 연산 종류별

In [ ]:
rows = []
for name, d in DATA.items():
    for ph, v in d["phases"].items():
        s = v["summary"]
        total = s["sum_of_ops_s"]
        rows.append({"result": name, "phase": ph,
                     **{k: f"{t / total * 100:.0f}%" for k, t in sorted(s["by_kind_s"].items())}})
pd.DataFrame(rows).set_index(["result", "phase"])

## 정리

1. **같은 matmul, 다른 병목**: prefill에서 compute-bound였던 7개 가중치 matmul이 decode에서는 AI ≈ 8로 떨어져 memory-bound가 된다. decode를 빠르게 하려면 FLOPs가 아니라 **읽는 바이트**를 줄여야 한다(batch 키우기, weight quantization).
2. **`qk`·`pv`는 prefill에서도 memory-bound**다(AI ≈ 114 < ridge). score 행렬을 HBM에 쓰고 다시 읽기 때문이다. fused attention(FlashAttention 계열)이 이 트래픽을 없앤다.
3. **softmax는 FLOPs가 거의 없는데도 prefill 시간의 1/4~1/3**을 쓴다. fusion의 첫 번째 대상이다.
4. **decode의 작은 연산은 `latency`로 판정**된다. 데이터가 수십 KB라 kernel을 띄우는 시간이 실제 작업보다 길다. CUDA graph나 `jax.jit`처럼 launch를 줄이는 방법이 효과적이다.

측정 방법과 코드는 [LLM roofline 노트북](llm_roofline.ipynb)과 [`llm_roofline/`](../llm_roofline)에 있다.